# Dataset Understanding — Stage 1: Structural Understanding

This notebook performs Stage 1 (Structural Understanding) of the Dataset Understanding Strategy: 
- row counts, 
- primary key uniqueness, 
- referential integrity between tables, 
- join feasibility, and 
- quarter consistency.

**Scope:** structural checks only. No feature engineering, no distributions, visualizations, or correlations, and no SQL/database work — those belong to later steps. Raw CSVs are loaded directly with pandas here for inspection only; this does not duplicate or replace `ingest.py`'s verification logic.

In [ ]:
import pandas as pd

RAW_DIR = "../data/raw"

demographics = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_demographics.csv")
location = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_location.csv")
population = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_population.csv")
services = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_services.csv")
status = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_status.csv")

tables = {
    "Demographics": demographics,
    "Location": location,
    "Population": population,
    "Services": services,
    "Status": status,
}

## 1. Row Counts

In [ ]:
EXPECTED_ROWS = {
    "Demographics": 7043,
    "Location": 7043,
    "Population": 1671,
    "Services": 7043,
    "Status": 7043,
}

for name, df in tables.items():
    n_rows, n_cols = df.shape
    expected = EXPECTED_ROWS[name]
    status_label = "PASS" if n_rows == expected else "FAIL"
    print(f"[{status_label}] {name}: {n_rows} rows, {n_cols} columns (expected {expected} rows)")

## 2. Primary Key Uniqueness

In [ ]:
pk_columns = {
    "Demographics": "Customer ID",
    "Location": "Customer ID",
    "Services": "Customer ID",
    "Status": "Customer ID",
    "Population": "Zip Code",
}

for name, key in pk_columns.items():
    df = tables[name]
    n_duplicates = int(df[key].duplicated().sum())
    status_label = "PASS" if n_duplicates == 0 else "FAIL"
    print(f"[{status_label}] {name}: {n_duplicates} duplicate '{key}' values")

## 3. Referential Integrity (Customer ID)

In [ ]:
demographics_ids = set(demographics["Customer ID"])

for name in ["Location", "Services", "Status"]:
    df = tables[name]
    table_ids = set(df["Customer ID"])

    missing_from_demographics = table_ids - demographics_ids
    missing_from_table = demographics_ids - table_ids

    status_label = "PASS" if not missing_from_demographics and not missing_from_table else "FAIL"
    print(f"[{status_label}] {name} <-> Demographics:")
    print(f"    {name} IDs not found in Demographics: {len(missing_from_demographics)}")
    print(f"    Demographics IDs not found in {name}: {len(missing_from_table)}")

## 4. Join Feasibility (Zip Code)

In [ ]:
population_zips = set(population["Zip Code"])
location_zips = location["Zip Code"]

unmatched_mask = ~location_zips.isin(population_zips)
n_unmatched = int(unmatched_mask.sum())

status_label = "PASS" if n_unmatched == 0 else "FAIL"
print(f"[{status_label}] Location rows with a Zip Code not found in Population: {n_unmatched}")

## 5. Quarter Consistency

In [ ]:
for name in ["Services", "Status"]:
    df = tables[name]
    value_counts = df["Quarter"].value_counts()
    n_distinct = df["Quarter"].nunique()
    status_label = "PASS" if n_distinct == 1 else "FAIL"
    print(f"[{status_label}] {name}: {n_distinct} distinct Quarter value(s)")
    print(value_counts.to_string())
    print()

## 6. Summary

Stage 1 structural checks cover: 
- row counts for all five raw tables against the expected spec used by `ingest.py`;
- `Customer ID` uniqueness in Demographics, Location, Services, and Status;
- `Zip Code` uniqueness in Population;
- Customer ID referential integrity between Demographics and each of Location, Services, and Status;
- Zip Code join feasibility between Location and Population; 
- Quarter consistency in Services and Status.

If every check above printed `PASS`, the five raw tables are structurally sound for downstream joining: unique keys, no orphaned foreign keys, and a single consistent Quarter. Any `FAIL` above should be treated as a finding to investigate before relying on that relationship in later stages.

# Dataset Understanding — Stage 2: Business Understanding

This section performs Stage 2 (Business Understanding) of the Dataset Understanding Strategy: 
- classifying each table's business grain and role,
- identifying the candidate target variable, 
- flagging leakage-prone columns,
- inspecting (not deciding) the voluntary-churn mapping, and 
- checking the dataset against the assumptions the locked Data Design made about it.

**Scope:** business-meaning review and classification only. No SQL, no feature engineering, no modeling. This section documents and inspects; it
does not decide the final voluntary-churn mapping — that mapping will be discussed separately before being implemented in `sql/03_target/`.

## Business Entity Classification

| Table | Grain | Role |
|---|---|---|
| Demographics | Customer-level | Static attributes |
| Location | Customer-level | Static attributes |
| Population | Zip-code-level | Reference / lookup table |
| Services | Customer-level, quarterly snapshot | Service usage & billing |
| Status | Customer-level, quarterly snapshot | Churn outcome & derived scores |

## Candidate Target Variable

In [ ]:
print("Customer Status value counts:")
print(status["Customer Status"].value_counts())
print()

print("Churn Label vs. Churn Value crosstab:")
print(pd.crosstab(status["Churn Label"], status["Churn Value"]))

`Churn Label` and `Churn Value` are consistent with each other (a clean Yes/No <-> 1/0 mapping with no crossover), and both are fully determined by `Customer Status` (`Churned` maps to `Yes`/`1`; every other status maps to `No`/`0`). 

**`Churn Value`** is the candidate raw target — it's already a numeric binary encoding that a modeling pipeline can consume directly, though the underlying business definition of "churn" it encodes still needs scrutiny (see the voluntary-churn mapping section below).

## Leakage-Prone Variable Identification

Columns that are only known *after* churn has already happened, and are not realistically available at scoring time for an active customer:

| Table | Column | Why it's leakage-prone |
|---|---|---|
| Status | `Churn Score` | An internal risk score computed as part of the churn/exit process, not an input to it |
| Status | `CLTV` | Customer Lifetime Value, likely computed retrospectively alongside the churn outcome |
| Status | `Churn Category` | Only populated for customers who have already churned |
| Status | `Churn Reason` | Only populated for customers who have already churned |
| Status | `Satisfaction Score` | Plausibly collected as part of an exit survey rather than ongoing — **needs verification**, not assumed here |

Additionally, `Count` columns appear in Demographics, Location, Services, and Status. These all appear to be reporting/dashboarding artifacts (a constant `1` per row, used for aggregation in the source BI tool) rather than genuine customer attributes, so they should be treated as **Identifier/Excluded** rather than **Feature**.

### Resolving the Satisfaction Score leakage flag

`Satisfaction Score` was flagged above as needing verification rather than assumed to be leakage. The empirical test: does it have real values for customers who *haven't* churned? If it's only populated for churned customers, it's exit-survey-only (leakage). If it's populated regardless of outcome, it's an ongoing sentiment metric (a legitimate feature).

In [ ]:
print("Satisfaction Score availability by Customer Status:")
print(status.groupby("Customer Status")["Satisfaction Score"].agg(
    non_null_count="count",
    total_count="size",
))

Checked empirically rather than assumed: `Satisfaction Score` is fully populated (no nulls) across all three `Customer Status` values — Stayed, Joined, and Churned alike. This means it's collected on an ongoing basis, not only as part of a post-churn exit survey.

**Conclusion: `Satisfaction Score` is NOT leakage.** It reflects a customer's sentiment while still active, and is genuinely available at scoring time — reclassified from "needs verification" to **Feature**.

## Preparing the Voluntary-Churn Mapping (inspection only — no decision yet)

This is inspection to inform a mapping decision that will be made separately. Nothing here classifies or maps values to voluntary/involuntary — it only surfaces the real `Churn Category` / `Churn Reason` values.

In [ ]:
churned = status[status["Customer Status"] == "Churned"]

print("Churn Category value counts (Churned customers only):")
print(churned["Churn Category"].value_counts())
print()

print("Churn Reason value counts (Churned customers only):")
print(churned["Churn Reason"].value_counts())

In [ ]:
pd.set_option("display.max_rows", None)
crosstab = pd.crosstab(churned["Churn Category"], churned["Churn Reason"])
crosstab = crosstab.loc[:, (crosstab != 0).any(axis=0)]
crosstab

## Verifying Dataset Selection Assumptions

- **Single-quarter snapshot, not longitudinal.** Stage 1 already confirmed `Services` and `Status` each report exactly one distinct `Quarter` value (Q3) across all rows. In business terms: this dataset is a single point-in-time snapshot of the customer base, not a time series of customer history, so temporal validation approaches (e.g. train on earlier quarters, test on later ones) are out of scope for this project.

- **Voluntary vs. involuntary approximation is genuinely relevant.** The Data Design anticipated a fallback where "voluntary churn" would need to be approximated from whatever fields are available, rather than being a purpose-built label. The fact that `Churn Category` and `Churn Reason` exist at all — and that their values (inspected above) mix reasons that read as voluntary (e.g. competitor offers, dissatisfaction) with reasons that read as involuntary or non-behavioral (e.g. moved, deceased) — confirms that this approximation fallback is genuinely needed here, not a hypothetical edge case.

### Voluntary-churn mapping — decided

The voluntary-churn mapping was decided outside this notebook and is recorded in ADR-0006. Summary: every Churned row is treated as voluntary except the 6 Deceased rows, which are excluded from the modeling dataset.

## Stage 2 Summary

**Classified:** 
- each table's business grain and role (customer-level static attributes, zip-level reference data, or quarterly snapshots of usage and
outcome); 
- the candidate raw target (`Churn Value`, backed by `Churn Label` and `Customer Status`); 
- leakage-prone columns in Status (`Churn Score`, `CLTV`, `Churn Category`, `Churn Reason`); 
- `Satisfaction Score`, initially flagged as needing verification, empirically confirmed as fully populated regardless of churn outcome — reclassified as a legitimate feature, not leakage;
- `Count` columns across tables as Identifier/Excluded rather than Feature.

**Confirmed:** the dataset is a single-quarter (Q3) snapshot, not longitudinal, and the voluntary-versus-involuntary churn approximation the Data Design anticipated is genuinely applicable to this data.

**Resolved after this notebook:** the voluntary-churn mapping from `Churn Category` / `Churn Reason` to a voluntary-churn label was decided in ADR-0006. Summary: every Churned row is treated as voluntary except the 6 Deceased rows, which are excluded from the modeling dataset.